In [0]:
dbutils.widgets.text("catalog", "scl")
catalog = dbutils.widgets.get("catalog")

#region

In [0]:
df = spark.table(f"{catalog}.bronze.region")

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("r_regionkey",trim("r_regionkey").cast("long")) \
    .withColumn("r_name",upper(trim("r_name"))) \
    .withColumn("r_comment",trim("r_comment")) 

df = df.drop("source_path","ingested_at")
df = df.drop_duplicates()

#invalid_Data
df = df.withColumn("validation_error",
            when(col("r_regionkey").isNull() ,"INVALID_PK")                               
            .when(col("r_name").isNull() ,"INVALID_NAME").otherwise(lit(None)))

valid_df = df.filter(col("validation_error").isNull())
quarantine_df = df.filter(col("validation_error").isNotNull())

valid_df = valid_df.drop("validation_error")

valid_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.region")
quarantine_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.region_quarantine")


In [0]:
spark.table(f"{catalog}.silver.region").count()

In [0]:
spark.table(f"{catalog}.silver.region_quarantine").count()

#nation

In [0]:
df = spark.table(f"{catalog}.bronze.nation")

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("n_nationkey",trim("n_nationkey").cast("long")) \
    .withColumn("n_regionkey",trim("n_regionkey").cast("long")) \
    .withColumn("n_name",upper(trim("n_name"))) \
    .withColumn("n_comment",trim("n_comment")) 


#take latest_ingested_at
df = df.orderBy(col("ingested_at").desc()).dropDuplicates(["n_nationkey"]).orderBy(col("ingested_at").asc())
df = df.drop("source_path","ingested_at")

#valid_region_key
valid_region_id = [r[0] for r in spark.table(f"{catalog}.silver.region").select("r_regionkey").collect()]

df = df.withColumn("validation_error",
            when(col("n_nationkey").isNull() ,"INVALID_PK")                               
            .when(col("n_regionkey").isNull() | ~ col("n_regionkey").isin(valid_region_id),"invalid_FK_REGION").otherwise(lit(None)))

valid_df = df.filter(col("validation_error").isNull())
quarantine_df = df.filter(col("validation_error").isNotNull())

valid_df = valid_df.drop("validation_error")

valid_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.nation")
quarantine_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.nation_quarantine")


In [0]:
valid_df.count()


In [0]:
quarantine_df.count()


#customer

In [0]:
customer_df = spark.table(f"{catalog}.bronze.customer") 

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

customer_df =customer_df.withColumn("c_custkey",trim("c_custkey").cast("long")) \
    .withColumn("c_name",upper(trim("c_name"))) \
    .withColumn("c_address",trim("c_address")) \
    .withColumn("c_nationkey",trim("c_nationkey").cast("long")) \
    .withColumn("c_acctbal",trim("c_acctbal").cast("decimal(18,2)")) \
    .withColumn("c_mktsegment",upper(trim("c_mktsegment"))) \
    .withColumn("c_phone",trim("c_phone")) \
    .withColumn("c_comment",trim("c_comment")) 


customer_df = customer_df.withColumn("rn", row_number().over(Window.partitionBy("c_custkey").orderBy(col("ingested_at").desc()))) \
    .filter(col("rn") == 1).drop("rn")

#valid_region_key
valid_nation_id = [r[0] for r in spark.table(f"{catalog}.silver.nation").select("n_nationkey").collect()]

customer_df = customer_df.withColumn("validation_error",
    when(col("c_custkey").isNull() ,"INVALID_PK")                           
    .when(col("c_nationkey").isNull() | ~col("c_nationkey").isin(valid_nation_id),"invalid_FK_NATION")
    .when(col("c_acctbal") < 0 , "invalid_amount")
    .otherwise(lit(None))
)
customer_df = customer_df.drop("source_path","ingested_at")

valid_df = customer_df.filter(col("validation_error").isNull())
quarantine_df = customer_df.filter(col("validation_error").isNotNull())

valid_df = valid_df.drop("validation_error")

valid_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.customer")
quarantine_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.customer_quarantine")


In [0]:
valid_df.count()

In [0]:
quarantine_df.count()

#supplier

In [0]:
df = spark.table(f"{catalog}.bronze.supplier")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = df.withColumn("s_suppkey",trim("s_suppkey").cast("long")) \
    .withColumn("s_name",upper(trim("s_name"))) \
    .withColumn("s_address",trim("s_address")) \
    .withColumn("s_nationkey",trim("s_nationkey").cast("long")) \
    .withColumn("s_acctbal",trim("s_acctbal").cast("decimal(18,2)")) \
    .withColumn("s_phone",trim("s_phone")) \
    .withColumn("s_comment",trim("s_comment")) 


#latest record
df = df.withColumn("rn", row_number().over(Window.partitionBy("s_suppkey").orderBy(col("ingested_at").desc()))) \
    .filter(col("rn") == 1).drop("rn")

#valid_region_key
valid_nation_id = [r[0] for r in spark.table(f"{catalog}.silver.nation").select("n_nationkey").collect()]

df = df.withColumn("validation_error",
    when(col("s_suppkey").isNull() ,"INVALID_PK")                           
    .when(col("s_nationkey").isNull() |  ~col("s_nationkey").isin(valid_nation_id),"INVALID_FK_NATION")
    .when(col("s_acctbal") < 0 , "invalid_amount")
    .otherwise(lit(None))
)
df = df.drop("source_path","ingested_at")

valid_df = df.filter(col("validation_error").isNull())
quarantine_df = df.filter(col("validation_error").isNotNull())

valid_df = valid_df.drop("validation_error")

valid_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.supplier")
quarantine_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.supplier_quarantine")


In [0]:
spark.table(f"{catalog}.silver.supplier").count()

In [0]:
quarantine_df.count()

#orders

In [0]:
order_df = spark.table(f"{catalog}.bronze.orders")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

order_df = order_df.withColumn("o_orderkey",trim("o_orderkey").cast("long")) \
    .withColumn("o_custkey",trim("o_custkey").cast("long")) \
    .withColumn("o_orderstatus",trim("o_orderstatus")) \
    .withColumn("o_totalprice",trim("o_totalprice").cast("decimal(18,2)")) \
    .withColumn("o_orderdate",to_date(col("o_orderdate"),"yyyy-MM-dd")) \
    .withColumn("o_orderpriority",trim("o_orderpriority")) \
    .withColumn("o_clerk",trim("o_clerk")) \
    .withColumn("o_shippriority",trim("o_shippriority").cast("int")) \
    .withColumn("o_comment",trim("o_comment")) 

order_df = order_df.drop("source_path","ingested_at")

#latest record
order_df = order_df.withColumn("rn", row_number().over(Window.partitionBy("o_orderkey").orderBy(col("o_orderdate").desc()))) \
    .filter(col("rn") == 1).drop("rn")

#valid_region_key
valid_customer_id = [r[0] for r in spark.table(f"{catalog}.silver.customer").select("c_custkey").collect()]

order_df = order_df.withColumn("validation_error",
    when(col("o_orderkey").isNull() ,"INVALID_PK")                           
    .when(col("o_custkey").isNull() | ~col("o_custkey").isin(valid_customer_id),"invalid_FK_CUSTOMER")
    .when(col("o_orderstatus").isNull() | ~col("o_orderstatus").isin(["F","O","P"]),"invalid_order_status")
    .when(col("o_totalprice") < 0 , "invalid_amount")
    .otherwise(lit(None))
)

valid_df = order_df.filter(col("validation_error").isNull())
quarantine_df = order_df.filter(col("validation_error").isNotNull())

valid_df = valid_df.drop("validation_error")

valid_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.orders")
quarantine_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.orders_quarantine")

In [0]:
valid_df.count()

In [0]:
quarantine_df.count()

#part

In [0]:
part_df = spark.table(f"{catalog}.bronze.part")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

part_df = part_df.withColumn("p_partkey",trim("p_partkey").cast("long")) \
    .withColumn("p_name",trim("p_name")) \
    .withColumn("p_mfgr",trim("p_mfgr")) \
    .withColumn("p_brand",trim("p_brand")) \
    .withColumn("p_type",upper(trim("p_type"))) \
    .withColumn("p_size",trim("p_size").cast("int")) \
    .withColumn("p_container",upper(trim("p_container"))) \
    .withColumn("p_retailprice",trim("p_retailprice").cast("decimal(18,2)")) \
    .withColumn("p_comment",trim("p_comment")) 

#latest record
part_df = part_df.withColumn("rn", row_number().over(Window.partitionBy("p_partkey").orderBy(col("ingested_at").desc()))) \
    .filter(col("rn") == 1).drop("rn")

#invalid
part_df = part_df.withColumn("validation_error",
    when(col("p_partkey").isNull() ,"INVALID_PK")                           
    .when(col("p_retailprice") < 0 , "invalid_amount")
    .otherwise(lit(None))
)

part_df = part_df.drop("ingested_at")
part_df = part_df.drop("source_path")

valid_df = part_df.filter(col("validation_error").isNull())
quarantine_df = part_df.filter(col("validation_error").isNotNull())

valid_df = valid_df.drop("validation_error")

valid_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.part")
quarantine_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.part_quarantine")




In [0]:
part_df.count()

In [0]:
quarantine_df.count()

#partsupp

In [0]:
partsupp_df = spark.table(f"{catalog}.bronze.partsupp")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

partsupp_df = partsupp_df.withColumn("ps_partkey",trim("ps_partkey").cast("long")) \
    .withColumn("ps_suppkey",trim("ps_suppkey").cast("long")) \
    .withColumn("ps_availqty",trim("ps_availqty").cast("int")) \
    .withColumn("ps_supplycost",trim("ps_supplycost").cast("double")) \
    .withColumn("ps_comment",trim("ps_comment")) 

#latest record
partsupp_df = partsupp_df.withColumn("rn", row_number().over(Window.partitionBy("ps_partkey","ps_suppkey").orderBy(col("ingested_at").desc()))) \
    .filter(col("rn") == 1).drop("rn")

#valid_keys
valid_part_key = [r[0] for r in spark.table(f"{catalog}.silver.part").select("p_partkey").collect()]
valid_supp_key = [r[0] for r in spark.table(f"{catalog}.silver.supplier").select("s_suppkey").collect()]

#invalid
partsupp_df = partsupp_df.withColumn("validation_error",
    when((col("ps_partkey").isNull() | ~col("ps_partkey").isin(valid_part_key)),"INVALID_PK") 
    .when(col("ps_suppkey").isNull() | ~col("ps_suppkey").isin(valid_supp_key),"invalid_FK_PART")
    .when(col("ps_availqty").isNull() | (col("ps_availqty") < 0),"invalid_quantity")
    .when(col("ps_supplycost").isNull() | (col("ps_supplycost") < 0),"invalid_amount")
    .otherwise(lit(None))
)

partsupp_df = partsupp_df.drop("ingested_at")
partsupp_df = partsupp_df.drop("source_path")

valid_df = partsupp_df.filter(col("validation_error").isNull())
quarantine_df = partsupp_df.filter(col("validation_error").isNotNull())

valid_df = valid_df.drop("validation_error")

valid_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.partsupp")
quarantine_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.partsupp_quarantine")


In [0]:
spark.table(f"{catalog}.silver.partsupp").count()

In [0]:
spark.table(f"{catalog}.silver.partsupp_quarantine").count()

#lineitem

In [0]:
lineitem_df = spark.table(f"{catalog}.bronze.lineitem")

In [0]:
lineitem_df.count()

In [0]:
valid_order_key = [r[0] for r in spark.table(f"{catalog}.silver.orders").select("o_orderkey").collect()]
valid_part_key = [r[0] for r in spark.table(f"{catalog}.silver.part").select("p_partkey").collect()]
valid_supp_key = [r[0] for r in spark.table(f"{catalog}.silver.supplier").select("s_suppkey").collect()]

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

lineitem_df = lineitem_df.withColumn("l_orderkey",trim("l_orderkey").cast("long")) \
    .withColumn("l_partkey",trim("l_partkey").cast("long")) \
    .withColumn("l_suppkey",trim("l_suppkey").cast("long")) \
    .withColumn("l_linenumber",trim("l_linenumber").cast("int")) \
    .withColumn("l_quantity",trim("l_quantity").cast("decimal(18,2)")) \
    .withColumn("l_extendedprice",trim("l_extendedprice").cast("decimal(18,2)")) \
    .withColumn("l_discount",trim("l_discount").cast("decimal(18,2)")) \
    .withColumn("l_tax",trim("l_tax").cast("decimal(18,2)")) \
    .withColumn("l_returnflag",trim("l_returnflag")) \
    .withColumn("l_linestatus",trim("l_linestatus")) \
    .withColumn("l_shipdate",to_date(trim("l_shipdate"),"yyyy-MM-dd")) \
    .withColumn("l_commitdate",to_date(trim("l_commitdate"),"yyyy-MM-dd")) \
    .withColumn("l_receiptdate",to_date(trim("l_receiptdate"),"yyyy-MM-dd")) \
    .withColumn("l_shipinstruct",upper(trim("l_shipinstruct"))) \
    .withColumn("l_shipmode",upper(trim("l_shipmode"))) \
    .withColumn("l_comment",trim("l_comment")) 

#latest record
lineitem_df = lineitem_df.withColumn("rn",row_number().over(Window.partitionBy("l_orderkey","l_linenumber").orderBy(desc("l_shipdate")))) \
    .filter(col("rn")==1).drop("rn")

# FK validation via broadcast left semi joins (much faster than .isin() with large lists)
valid_orders = spark.table(f"{catalog}.silver.orders").select("o_orderkey").distinct()
valid_parts = spark.table(f"{catalog}.silver.part").select("p_partkey").distinct()
valid_suppliers = spark.table(f"{catalog}.silver.supplier").select("s_suppkey").distinct()

order_match = lineitem_df.join(valid_orders, lineitem_df["l_orderkey"] == valid_orders["o_orderkey"], "left") \
    .withColumn("_order_valid", when(col("o_orderkey").isNotNull(), lit(1)).otherwise(lit(0))) \
    .drop("o_orderkey")

part_match = order_match.join(valid_parts, order_match["l_partkey"] == valid_parts["p_partkey"], "left") \
    .withColumn("_part_valid", when(col("p_partkey").isNotNull(), lit(1)).otherwise(lit(0))) \
    .drop("p_partkey")

supp_match = part_match.join(valid_suppliers, part_match["l_suppkey"] == valid_suppliers["s_suppkey"], "left") \
    .withColumn("_supp_valid", when(col("s_suppkey").isNotNull(), lit(1)).otherwise(lit(0))) \
    .drop("s_suppkey")

lineitem_df = supp_match.withColumn("validation_error",
    when(col("l_orderkey").isNull(),"INVALID_PK")
    .when(col("_order_valid") == 0,"INVALID_PK")
    .when(col("l_partkey").isNull() | (col("_part_valid") == 0),"invalid_FK_PART")
    .when(col("l_suppkey").isNull() | (col("_supp_valid") == 0),"invalid_FK_SUPP")
    .when(col("l_linenumber").isNull() | (col("l_linenumber") < 0),"INVALID_PK")
    .when(col("l_quantity").isNull() | (col("l_quantity") <= 0),"invalid_quantity")
    .when(col("l_extendedprice").isNull() | (col("l_extendedprice") <= 0),"invalid_amount")
    .when(col("l_discount").isNull() | ((col("l_discount") < 0) | (col("l_discount") > 0.1)),"invalid_discount")
    .when(col("l_tax").isNull() | ((col("l_tax") < 0) | (col("l_tax") > 0.08)),"invalid_tax")
    .when(col("l_returnflag").isNull() | ~col("l_returnflag").isin(["A","R","N"]),"invalid_returnflag")
    .when(col("l_linestatus").isNull() | (~col("l_linestatus").isin(["F","O"])),"invalid_linestatus")
    .when((col("l_shipdate") < to_date(lit("1992-01-01"),"yyyy-MM-dd")),"invalid_shipdate")
    .when((col("l_shipdate") > to_date(lit("1998-12-31"),"yyyy-MM-dd")),"invalid_shipdate")
    .when((col("l_receiptdate") < to_date(lit("1992-01-01"),"yyyy-MM-dd")),"invalid_receiptdate")
    .when((col("l_receiptdate") > to_date(lit("1998-12-31"),"yyyy-MM-dd")),"invalid_receiptdate")
    .when((col("l_commitdate") < to_date(lit("1992-01-01"),"yyyy-MM-dd")),"invalid_commitdate")
    .when((col("l_commitdate") > to_date(lit("1998-12-31"),"yyyy-MM-dd")),"invalid_commitdate")
    .otherwise(lit(None))
)

lineitem_df = lineitem_df.drop("ingested_at", "source_path", "_order_valid", "_part_valid", "_supp_valid")

valid_df = lineitem_df.filter(col("validation_error").isNull())
quarantine_df = lineitem_df.filter(col("validation_error").isNotNull())

valid_df = valid_df.drop("validation_error")

# valid_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.lineitem")
# quarantine_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.lineitem_quarantine")


In [0]:
valid_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.lineitem")
quarantine_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.lineitem_quarantine")

In [0]:
spark.table(f"{catalog}.silver.lineitem").count()

In [0]:
spark.table(f"{catalog}.silver.lineitem_quarantine").count()